# 17.2 `pyproject.toml` — One File for Everything

**Prerequisites:** 17.1 Environments, 15.6 Testing in Practice, 16.6 Adopting Types  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why `setup.py` became `pyproject.toml`, and what each section is for
- `[build-system]` — the two lines that make a directory buildable
- `[project]` — the standard metadata every tool reads (PEP 621)
- Dependencies and **optional-dependency groups**
- `[project.scripts]` — turning a function into a command
- 🔴 One file configuring `pytest`, `coverage`, `mypy` and `ruff` together
- Reading it yourself with **`tomllib`** (3.11+, no dependency)
- Single-sourcing the version
- 🔴 TOML gotchas that cost an afternoon

---

## One file, finally

For twenty years a Python project's metadata lived in **`setup.py`** — an executable script that
had to be *run* to find out what your package was called. That made it impossible to inspect a
package safely, and every tool grew its own config file:

```
   setup.py          .flake8          pytest.ini
   setup.cfg         .isort.cfg       .coveragerc
   MANIFEST.in       mypy.ini         tox.ini
```

**`pyproject.toml`** (PEPs 517, 518 and 621) replaced the lot. It is **declarative data**, not
code, so a tool can read your project's name and dependencies without executing anything.

| Section | Defined by | Holds |
|---|---|---|
| `[build-system]` | PEP 518 | how to build this project |
| `[project]` | PEP 621 | 🔴 name, version, dependencies — the standard metadata |
| `[tool.NAME]` | each tool | that tool's own settings |
| `[dependency-groups]` | PEP 735 | development dependency groups (2024) |

You have already met `[tool.*]` three times in this curriculum — pytest in **15.3**, coverage in
**15.6**, mypy in **16.6**. This notebook is the whole file.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
import tomllib
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py172_"))
PROJECT = WORK / "jobkit"


def write(rel, source, root=None):
    path = (root or PROJECT) / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def run(args, cwd=None, timeout=600, label=None):
    done = subprocess.run(args, cwd=cwd or PROJECT, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=timeout)
    shown = label or " ".join(
        "python" if a == sys.executable else str(a) for a in args)
    body = (done.stdout + done.stderr).strip() or "(no output)"
    return (f"$ {shown}\n" + "-" * 68 + "\n" + body
            + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## A complete, real `pyproject.toml`

Every line below does something. We build the file first, then take it apart.

In [ ]:
write("pyproject.toml", r"""
    # ---- how to build this project -------------------------------------
    [build-system]
    requires = ["setuptools>=77"]
    build-backend = "setuptools.build_meta"

    # ---- what this project IS (PEP 621) --------------------------------
    [project]
    name = "jobkit"
    version = "0.1.0"
    description = "Retry policies and job scheduling helpers"
    readme = "README.md"
    requires-python = ">=3.12"
    license = "MIT"
    authors = [{name = "Aditya Tripathi"}]
    keywords = ["retry", "backoff", "jobs"]
    classifiers = [
        "Programming Language :: Python :: 3.12",
        "Typing :: Typed",
    ]

    dependencies = [
        "httpx>=0.27,<1",
    ]

    [project.optional-dependencies]
    dev = ["pytest>=8", "coverage[toml]>=7", "mypy>=1.8", "ruff>=0.5"]
    postgres = ["psycopg[binary]>=3.1"]

    [project.urls]
    Homepage = "https://example.com/jobkit"
    Source = "https://example.com/jobkit/src"

    # ---- turn a function into a command --------------------------------
    [project.scripts]
    jobkit = "jobkit.cli:main"

    # ---- where the code lives ------------------------------------------
    [tool.setuptools.packages.find]
    where = ["src"]

    # ---- every tool's settings, in one place ---------------------------
    [tool.pytest.ini_options]
    testpaths = ["tests"]
    addopts = "-ra --strict-markers --strict-config"
    filterwarnings = ["error"]
    markers = ["slow: excluded from the pre-commit run"]

    [tool.coverage.run]
    branch = true
    source = ["src"]

    [tool.coverage.report]
    fail_under = 80
    show_missing = true

    [tool.mypy]
    strict = true

    [tool.ruff]
    line-length = 100
    target-version = "py312"

    [tool.ruff.lint]
    select = ["E", "F", "W", "I", "B", "UP", "SIM"]
""")

write("README.md", "# jobkit\n\nRetry policies and job scheduling helpers.\n")
write("src/jobkit/__init__.py", '__version__ = "0.1.0"\n')
write("src/jobkit/py.typed", "")
write("src/jobkit/retry.py", r"""
    def retry_delay(attempt: int, base: float = 1.0, ceiling: float = 30.0) -> float:
        delay = base
        for _ in range(attempt):
            delay *= 2
        return min(delay, ceiling)
""")
write("src/jobkit/cli.py", r"""
    import sys

    from jobkit.retry import retry_delay


    def main() -> int:
        attempt = int(sys.argv[1]) if len(sys.argv) > 1 else 0
        print(f"{retry_delay(attempt):.1f}")
        return 0
""")
write("tests/test_retry.py", r"""
    import pytest

    from jobkit.retry import retry_delay


    @pytest.mark.parametrize("attempt,expected", [(0, 1.0), (1, 2.0), (9, 30.0)])
    def test_backoff(attempt: int, expected: float) -> None:
        assert retry_delay(attempt) == expected
""")

print("project created:")
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print("   ", path.relative_to(PROJECT).as_posix())

## `[build-system]` — the two lines that matter most

```toml
[build-system]
requires = ["setuptools>=77"]
build-backend = "setuptools.build_meta"
```

This says: *to build this project, install `setuptools`, then ask it to do the work*. A build
frontend (`pip`, `build`) creates an isolated environment, installs `requires`, and calls the
backend.

> **Version note.** The floor is doing work: `license = "MIT"` as a bare SPDX string
> (PEP 639, used in `[project]` above) is only understood by **setuptools ≥ 77** — older
> setuptools rejects the plain string, expecting the pre-639 `{text = "..."}` table. If you
> copy the modern `[project]` table, copy the modern floor with it.

🔴 **Without a `[build-system]` table, tools fall back to legacy `setup.py` behaviour.** Always
include it, even if the rest of the file is short.

| Backend | `build-backend` | Notes |
|---|---|---|
| setuptools | `setuptools.build_meta` | the default; most compatible |
| hatchling | `hatchling.build` | modern, clean defaults |
| flit | `flit_core.buildapi` | minimal, pure-Python packages |
| poetry | `poetry.core.masonry.api` | if you use poetry |
| maturin | `maturin` | Rust extensions |

The choice barely affects the `[project]` table, which is standardised.

## Reading it with `tomllib`

TOML parsing is in the standard library since **3.11** — no dependency needed. This is how you
single-source your version, or write a script that checks your own metadata.

In [ ]:
data = tomllib.loads((PROJECT / "pyproject.toml").read_text(encoding="utf-8"))

project = data["project"]
print("name          :", project["name"])
print("version       :", project["version"])
print("requires-python:", project["requires-python"])
print("dependencies  :", project["dependencies"])
print()
print("optional groups:")
for group, packages in project["optional-dependencies"].items():
    print(f"   {group:10} {packages}")
print()
print("console scripts:", project["scripts"])
print("tool tables    :", sorted(data["tool"]))
print()
print("🔴 tomllib.load() needs a BINARY file object - open(path, 'rb').")
print("   tomllib.loads() takes a str. There is no dump; use `tomli-w` to write.")

## Dependencies

```toml
dependencies = [
    "httpx>=0.27,<1",              # runtime: installed with the package
]

[project.optional-dependencies]
dev = ["pytest>=8", "mypy>=1.8"]   # extras: installed on request
postgres = ["psycopg[binary]>=3.1"]
```

| Install command | Gets |
|---|---|
| `pip install jobkit` | `jobkit` + `httpx` |
| `pip install jobkit[dev]` | …plus pytest and mypy |
| `pip install jobkit[dev,postgres]` | …plus psycopg |
| `pip install -e ".[dev]"` | 🔴 the usual local development command |

🔴 **`dependencies` are what your *users* need.** Test and lint tools are not. Putting `pytest`
in the main list means everyone who installs your library also installs pytest — a common and
irritating mistake.

> **`[dependency-groups]` (PEP 735, 2024)** is the newer home for purely-local groups like
> `dev`. It is not published in package metadata at all, which is more correct than an extra.
> Support is still spreading; `optional-dependencies` remains the compatible choice.

### `coverage[toml]` — what the brackets mean

An **extra** on a dependency. `coverage[toml]` means "coverage, plus whatever it needs to read
TOML config". You will see `psycopg[binary]`, `requests[security]`, `uvicorn[standard]` — all
the same mechanism.

## `[project.scripts]` — a function becomes a command

```toml
[project.scripts]
jobkit = "jobkit.cli:main"
   ▲          ▲        ▲
   │          │        └── the function to call
   │          └── the module it lives in
   └── the command name that appears on PATH
```

On install, pip generates a small executable named `jobkit` that imports `jobkit.cli` and calls
`main()`. That is exactly how `pytest`, `ruff` and `pip` itself got onto your `PATH` — you saw
those executables in `Scripts/` in **17.1**.

The function should return an `int` exit code and take no arguments.

## 🔴 One file, every tool

The strongest argument for `pyproject.toml` is that your editor, your terminal and CI all read
the *same* settings. The next cell runs three tools against the project — each picking up its
own `[tool.*]` table with no command-line flags at all.

In [ ]:
print(run([sys.executable, "-m", "ruff", "check", "src", "tests",
           "--output-format=concise"],
          label="ruff check src tests        # reads [tool.ruff]"))
print()
print(run([sys.executable, "-m", "mypy", "src",
           "--cache-dir", str(WORK / ".mypy_cache"),
           "--no-color-output", "--no-error-summary"],
          label="mypy src                    # reads [tool.mypy]"))

Two tools, zero flags, settings from one file. `mypy` ran under `strict = true`
because the file said so — not because anyone remembered to type `--strict`.

> The `pytest` and `coverage` tables work the same way; **15.3** and **15.6** cover what those
> settings mean. Running them here would need the package installed, which is **17.3**.

## Single-sourcing the version

The version appears in `pyproject.toml` and usually in `__init__.py` too. Two copies drift —
and yes, the demo project above commits exactly this sin, typing `0.1.0` in both
`pyproject.toml` and `src/jobkit/__init__.py`; option 2 below is the cure it should adopt.
Three approaches, in order of preference:

**1. Read it from the installed metadata** — the version lives only in `pyproject.toml`:

```python
from importlib.metadata import version
__version__ = version("jobkit")
```

**2. Let the backend read it from your code** — the version lives only in `__init__.py`:

```toml
[project]
dynamic = ["version"]

[tool.setuptools.dynamic]
version = {attr = "jobkit.__version__"}
```

**3. Derive it from git tags** — with `setuptools-scm` or `hatch-vcs`; the tag is the version.

🔴 What you must not do is type it in two places and rely on remembering.

In [ ]:
print("importlib.metadata reads the INSTALLED package's metadata:")
print()
from importlib.metadata import metadata, version

for package in ("pytest", "mypy", "ruff"):
    info = metadata(package)
    print(f"   {package:8} {version(package):10} requires-python "
          f"{info.get('Requires-Python', '(unset)')}")

print()
print("🔴 This reads what pip installed, NOT your pyproject.toml -")
print("   so it only works for a package that is actually installed (17.3).")

## 🔴 TOML gotchas

TOML looks obvious and has four sharp edges.

**1. Strings must be double-quoted.** `'single'` works for *literal* strings only, and
`name = bare` is a syntax error.

**2. `[[double.brackets]]` is an array of tables**, not a typo. It is how you write repeated
sections — mypy's per-module overrides (**16.6**) use exactly this:

```toml
[[tool.mypy.overrides]]
module = "app.legacy"
disallow_untyped_defs = false

[[tool.mypy.overrides]]      # a second entry in the same array
module = "vendor.*"
ignore_missing_imports = true
```

**3. Everything after a table header belongs to that table.** A key placed below
`[tool.ruff.lint]` is a *lint* setting, even if you meant it for `[tool.ruff]`. This is the
single most common `pyproject.toml` bug, and how it fails depends on the tool: some validate
their tables and **refuse to run** over a field they do not recognise (ruff), while others
**carry on**, demoting the mistake to a warning you may never read (pytest, unless
`--strict-config` promotes it to an error). The next cell shows both.

**4. Booleans are lowercase.** `true`, not `True`. TOML is not Python.

In [ ]:
BROKEN = WORK / "broken"
write("pyproject.toml", r"""
    [tool.ruff]
    target-version = "py312"

    [tool.ruff.lint]
    select = ["E", "F"]
    line-length = 100          # 🔴 meant for [tool.ruff], landed in [tool.ruff.lint]
""", root=BROKEN)
write("demo.py", "x = 1\n", root=BROKEN)

parsed = tomllib.loads((BROKEN / "pyproject.toml").read_text(encoding="utf-8"))
print("what the file actually says:")
print("   [tool.ruff]      ->", parsed["tool"]["ruff"].get("line-length", "(not set!)"))
print("   [tool.ruff.lint] ->", parsed["tool"]["ruff"]["lint"].get("line-length"))
print()

# ---- the loud failure: ruff validates its tables -------------------
print(run([sys.executable, "-m", "ruff", "check", "."], cwd=BROKEN,
          label="ruff check .                # 🔴 ruff refuses to run"))
print()

# ---- the quiet failure: pytest keeps going -------------------------
QUIET = WORK / "quiet"
write("pyproject.toml", r"""
    [tool.pytest.ini_options]
    line-length = 100          # 🔴 same mistake, pytest's table this time
""", root=QUIET)
write("test_nothing.py", "def test_ok():\n    assert True\n", root=QUIET)

print(run([sys.executable, "-m", "pytest", "-q"], cwd=QUIET,
          label="pytest -q                   # the suite passes anyway"))
print()
print(run([sys.executable, "-m", "pytest", "-q", "--strict-config"], cwd=QUIET,
          label="pytest -q --strict-config  # the flag that makes it loud"))
print()

print("And the types TOML gives you back (gotcha 4):")
for value in ("true", "True"):
    try:
        result = tomllib.loads(f"flag = {value}")
        print(f"   flag = {value:5} -> {result['flag']!r}")
    except tomllib.TOMLDecodeError as exc:
        print(f"   flag = {value:5} -> TOMLDecodeError: {exc}")


Same mistake, two very different mornings:

- **ruff refused to run** — "unknown field 'line-length'", exit code 2. Loud is the good
  case: the config cannot stay wrong, because nothing works until it is fixed.
- **pytest ran the suite green** and filed a `PytestConfigWarning` in the summary — one line
  that scrolls past. Without `--strict-config`, a misspelled or misplaced pytest option
  simply does nothing, forever. That flag is why the demo project's `addopts` includes it.

And `True` is a `TOMLDecodeError` while `true` is a boolean — TOML is not Python.

> **How to catch this class of bug:** never assume which kind of tool you are holding —
> *verify the config is actually read*. Most tools can print the settings they resolved
> (`ruff check --show-settings`, `mypy --verbose`), and strictness flags like
> `pytest --strict-config` turn quiet mistakes into loud ones. When a setting "does not
> work", print what the tool believes before changing anything (**15.9**: check your
> assumptions, do not guess).

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Putting a key under the wrong table.** `[tool.ruff.lint]` and `[tool.ruff]` are different tables. ruff fails loudly on the misplaced key; pytest only warns and carries on unless `--strict-config` — know which kind of tool you are holding.
2. **Omitting `[build-system]`.** Tools then fall back to legacy `setup.py` behaviour.
3. 🔴 **Putting `pytest` and `mypy` in `dependencies`.** Everyone installing your library then installs your test tools. They belong in an optional group.
4. **Writing `True` instead of `true`.** TOML booleans are lowercase, and this is a parse error rather than a silent one — the good case.
5. **Typing the version in two places** and letting them drift. Single-source it.
6. **Pinning exact versions in `dependencies`** for a library — see **17.1**.
7. **Using `tomllib.load()` with a text-mode file object.** It requires binary mode.
8. **Expecting `tomllib` to write TOML.** It is read-only; use `tomli-w` if you must write.
9. **Assuming a setting works because the tool did not complain.** Some tools reject unknown keys loudly, others demote them to a warning or ignore them — print the resolved settings instead of trusting silence.

## Best Practices

- Keep one `pyproject.toml` and delete `setup.py`, `setup.cfg`, `pytest.ini` and friends.
- Always declare `[build-system]`, even for a small project.
- Split runtime dependencies from development ones with `optional-dependencies`.
- Use ranges for a library, and keep `requires-python` honest.
- Single-source the version — `importlib.metadata.version()` or a `dynamic` entry.
- Configure every tool in `[tool.*]` so editor, terminal and CI agree (**15.6**, **16.6**).
- Read the file with `tomllib` when you need to script against your own metadata.
- When a setting seems ignored, ask the tool what it resolved before changing anything.

## Practice Exercises

Try these before moving on.

1. Write a minimal `pyproject.toml` with only `[build-system]` and `[project]`. What is the smallest file that still builds?
2. Add a `dev` optional-dependency group and install it with `pip install -e '.[dev]'`. What appears in the environment that would not otherwise?
3. 🔴 Deliberately put `line-length` under `[tool.ruff.lint]` and run `ruff check` — read the error and its exit code (`echo $?` / `$LASTEXITCODE`). Then misspell an option under `[tool.pytest.ini_options]` and run `pytest` twice, with and without `--strict-config`. Which of the three failure modes would you rather debug?
4. Move a `pytest.ini` and a `mypy.ini` from a project of yours into `pyproject.toml`. Did everything keep working?
5. Single-source the version with `importlib.metadata.version()`, then bump it in `pyproject.toml` and confirm `__version__` follows.
6. Add a `[project.scripts]` entry, install the package, and find the generated executable in your venv's `Scripts/` or `bin/` (**17.1**).
7. Write a script using `tomllib` that reads your `pyproject.toml` and fails if any dependency has no upper bound. That is a real CI check.
8. **Interview question:** why was `setup.py` replaced by a declarative file, and what does that make possible that was not possible before?

---

## Version notes

| Version | Change |
|---|---|
| **PEP 735 (2024)** | `[dependency-groups]` — local-only groups, not published as extras |
| **PEP 639 (2024)** | `license = "MIT"` as a plain SPDX string, replacing the license classifiers — needs setuptools ≥ 77 |
| **3.11** | 🔴 **`tomllib` in the standard library** — reading `pyproject.toml` needs no dependency |
| **PEP 621 (2020)** | `[project]` standardised, so every backend reads the same metadata |
| **PEP 517/518 (2015–17)** | `[build-system]` and pluggable backends — the end of `setup.py` as an interface |

## Where next

| Notebook | Covers |
|---|---|
| **17.3** | building a wheel from this project and publishing it |
| **17.4** | `ruff` — the linter and formatter configured above |
| **17.5** | profiling and performance |

The folder index and a one-sentence summary are at the end of **17.5**.

## Related

- **17.1 Environments** — where `pip install -e ".[dev]"` puts things
- **15.3 / 15.6** — the `[tool.pytest.ini_options]` and `[tool.coverage.*]` tables
- **16.6 Adopting Types** — the `[tool.mypy]` table and its per-module overrides
- **07 Module and Packages** — packages, which `[tool.setuptools.packages.find]` locates